# Validate Test Cases
Run `Model/pipeline.py` against `images/test_cases.json` and score against ground truth.

In [ ]:
import json, sys
from pathlib import Path

REPO = Path.cwd()
MODEL_DIR = REPO / 'Model'
IMG_DIR = REPO / 'images'
TESTS = IMG_DIR / 'test_cases.json'
OUT_DIR = REPO / 'out_validate'
OUT_DIR.mkdir(exist_ok=True)

if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))

from pipeline import EdgePipeline, draw, COCO
print('pipeline imported. COCO classes:', len(COCO))

In [ ]:
cases = json.loads(TESTS.read_text())
for c in cases:
    gt = c.get('ground_truth') or c.get('ground_truth_bbox')
    c['gt'] = gt
print(f'{len(cases)} test cases loaded')
cases[:2]

In [ ]:
pipe = EdgePipeline()

In [ ]:
# GT normalization: handle aliases (e.g. 'table' -> 'dining table')
ALIAS = {
    'table': 'dining table',
    'tv monitor': 'tv',
    'cellphone': 'cell phone',
}

def gt_match(pred_name: str, gt: str) -> bool:
    if not gt:
        return False
    g = gt.strip().lower()
    g = ALIAS.get(g, g)
    return pred_name.strip().lower() == g

results = []
for c in cases:
    img = (REPO / c['image_file']).resolve()
    if not img.exists():
        print(f"[skip] test {c['test_id']} missing image: {img}")
        results.append({**c, 'pred': None, 'ok': False, 'reason': 'missing image'})
        continue
    print(f"\n=== test {c['test_id']}: {c['prompt']!r} gt={c['gt']!r} img={img.name}")
    best, ranked, winners = pipe.run(img, c['prompt'], conf=0.05)
    if best is None:
        results.append({**c, 'pred': None, 'ok': False, 'reason': 'no detections'})
        continue
    out_path = OUT_DIR / f"test{c['test_id']:02d}_{img.stem}.jpg"
    draw(img, winners, c['prompt'], out_path)
    ok = gt_match(best['name'], c['gt'])
    results.append({
        'test_id': c['test_id'], 'prompt': c['prompt'], 'gt': c['gt'],
        'pred': best['name'], 'score': best['score'], 'sem': best['sem'], 'conf': best['conf'],
        'out': str(out_path), 'ok': ok,
    })
    print(f"  -> pred={best['name']} ok={ok}")

In [ ]:
# Summary table
passed = sum(1 for r in results if r.get('ok'))
total = len(results)
print(f'PASS {passed}/{total} = {passed/total*100:.1f}%')
print()
print(f"{'id':>3} {'ok':>3} {'gt':<14} {'pred':<14} {'score':>6}  prompt")
for r in results:
    print(f"{r['test_id']:>3} {('Y' if r.get('ok') else 'N'):>3} {str(r.get('gt','')):<14} {str(r.get('pred','')):<14} {r.get('score',0):>6.3f}  {r.get('prompt','')}")

In [ ]:
# Show annotated outputs inline
from IPython.display import Image as IPyImage, display, Markdown
for r in results:
    if r.get('out'):
        display(Markdown(f"**test {r['test_id']}** — gt=`{r['gt']}` pred=`{r['pred']}` ok=`{r['ok']}`"))
        display(IPyImage(r['out']))

In [ ]:
# Dump results json
report = OUT_DIR / 'results.json'
report.write_text(json.dumps(results, indent=2))
print('wrote', report)